In [4]:
!pip install torch torchaudio librosa numpy soundfile matplotlib

  Using cached torch-2.8.0-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached torchaudio-2.8.0-cp313-cp313-win_amd64.whl.metadata (7.2 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached matplotlib-3.10.6-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
Using cached torch-2.8.0-cp313-cp313-win_amd64.whl (241.3 MB)
Using cached torchaudio-2.8.0-cp313-cp313-win_amd64.whl (2.5 MB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl (1.0 MB)
Using cached matplotlib-3.10.6-cp313-cp313-win_amd64.whl (8.1 MB)
Using cached pooch-1.8.2-py3-none-any.whl (64 kB)
Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl (8.7 MB)

   ---------------------------------------- 0/7 [torch]
   ---------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Projects\\MachineLearning\\.venv\\Lib\\site-packages\\torch\\_dynamo\\variables\\optimizer.py'
Check the permissions.



In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpeakerEncoder(nn.Module):
    def __init__(self, mel_channels=80, hidden_dim=256, embedding_dim=256):
        super().__init__()
        
        # Mel-spectrogram preprocessing
        self.pre_net = nn.Sequential(
            nn.Linear(mel_channels, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        
        # LSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=3,
            batch_first=True,
            bidirectional=True
        )
        
        # Project to speaker embedding
        self.projection = nn.Linear(hidden_dim * 2, embedding_dim)
        
    def forward(self, mel_spectrograms):
        # mel_spectrograms: (batch, time, mel_channels)
        x = self.pre_net(mel_spectrograms)
        
        # LSTM processing
        self.lstm.flatten_parameters()
        x, _ = self.lstm(x)
        
        # Temporal average pooling
        x = x.mean(dim=1)
        
        # L2 normalize embeddings
        x = F.normalize(self.projection(x), p=2, dim=1)
        return x

In [3]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=512, encoder_dim=512):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv_layers = nn.Sequential(
            nn.Conv1d(embedding_dim, encoder_dim, 5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(encoder_dim),
            nn.Conv1d(encoder_dim, encoder_dim, 5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(encoder_dim),
        )
        self.lstm = nn.LSTM(encoder_dim, encoder_dim//2, batch_first=True, bidirectional=True)
        
    def forward(self, text):
        x = self.embedding(text)  # (batch, text_len, embedding_dim)
        x = x.transpose(1, 2)  # (batch, embedding_dim, text_len)
        x = self.conv_layers(x)
        x = x.transpose(1, 2)  # (batch, text_len, encoder_dim)
        x, _ = self.lstm(x)
        return x

class Attention(nn.Module):
    def __init__(self, query_dim, key_dim, attention_dim=128):
        super().__init__()
        self.query_layer = nn.Linear(query_dim, attention_dim, bias=False)
        self.key_layer = nn.Linear(key_dim, attention_dim, bias=False)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        
    def forward(self, query, keys):
        # query: (batch, decoder_dim)
        # keys: (batch, text_len, encoder_dim)
        
        # Expand query to match keys
        query = query.unsqueeze(1)  # (batch, 1, decoder_dim)
        
        # Compute attention scores
        energy = torch.tanh(self.query_layer(query) + self.key_layer(keys))
        attention_scores = F.softmax(self.v(energy).squeeze(-1), dim=1)
        
        # Context vector
        context = torch.bmm(attention_scores.unsqueeze(1), keys).squeeze(1)
        return context, attention_scores

class Decoder(nn.Module):
    def __init__(self, mel_channels=80, encoder_dim=512, decoder_dim=1024, speaker_dim=256):
        super().__init__()
        
        self.prenet = nn.Sequential(
            nn.Linear(mel_channels, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        
        # Concatenate: prenet_out + context + speaker_embedding
        lstm_input_dim = 128 + encoder_dim + speaker_dim
        
        self.lstm1 = nn.LSTMCell(lstm_input_dim, decoder_dim)
        self.lstm2 = nn.LSTMCell(decoder_dim, decoder_dim)
        
        self.attention = Attention(decoder_dim, encoder_dim)
        
        self.mel_proj = nn.Linear(decoder_dim, mel_channels)
        self.stop_token = nn.Linear(decoder_dim, 1)
        
    def forward(self, encoder_outputs, speaker_embedding, mel_target=None):
        batch_size = encoder_outputs.size(0)
        
        # Initialize states
        h1, c1 = torch.zeros(batch_size, 1024), torch.zeros(batch_size, 1024)
        h2, c2 = torch.zeros(batch_size, 1024), torch.zeros(batch_size, 1024)
        
        # First decoder input (zeros)
        prev_mel = torch.zeros(batch_size, 80)
        
        mels, alignments = [], []
        
        for i in range(200 if mel_target is None else mel_target.size(1)):
            # Prenet
            prenet_out = self.prenet(prev_mel)
            
            # Attention
            context, attention_weights = self.attention(h2, encoder_outputs)
            
            # Concatenate inputs
            decoder_input = torch.cat([prenet_out, context, speaker_embedding], dim=1)
            
            # LSTM cells
            h1, c1 = self.lstm1(decoder_input, (h1, c1))
            h2, c2 = self.lstm2(h1, (h2, c2))
            
            # Output projections
            mel_frame = self.mel_proj(h2)
            stop_token = torch.sigmoid(self.stop_token(h2))
            
            mels.append(mel_frame)
            alignments.append(attention_weights)
            
            # Teacher forcing or autoregressive
            prev_mel = mel_target[:, i] if mel_target is not None else mel_frame
            
        return torch.stack(mels, dim=1), torch.stack(alignments, dim=1)

In [4]:
class DilatedConvBlock(nn.Module):
    def __init__(self, residual_channels, dilation):
        super().__init__()
        self.dilated_conv = nn.Conv1d(
            residual_channels, 
            2 * residual_channels, 
            kernel_size=3, 
            padding=dilation, 
            dilation=dilation
        )
        self.residual_proj = nn.Conv1d(residual_channels, residual_channels, 1)
        self.skip_proj = nn.Conv1d(residual_channels, residual_channels, 1)
        
    def forward(self, x):
        original = x
        x = self.dilated_conv(x)
        gate, filter = torch.chunk(x, 2, dim=1)
        x = torch.sigmoid(gate) * torch.tanh(filter)
        
        residual = self.residual_proj(x)
        skip = self.skip_proj(x)
        
        return (original + residual) * 0.707, skip

class WaveNetVocoder(nn.Module):
    def __init__(self, mel_channels=80, residual_channels=256, skip_channels=256, num_layers=30):
        super().__init__()
        
        # Conditioning network
        self.cond_net = nn.Conv1d(mel_channels, residual_channels, 3, padding=1)
        
        # Input projection
        self.input_proj = nn.Conv1d(1, residual_channels, 1)
        
        # Dilated convolution blocks
        self.dilated_blocks = nn.ModuleList()
        for i in range(num_layers):
            dilation = 2 ** (i % 10)  # Cycle through dilations
            self.dilated_blocks.append(DilatedConvBlock(residual_channels, dilation))
        
        # Output network
        self.out_net = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, 1),
            nn.ReLU(),
            nn.Conv1d(skip_channels, 256, 1)  # 256 for mu-law quantization
        )
        
    def forward(self, audio, mel_spectrogram):
        # audio: (batch, 1, time)
        # mel_spectrogram: (batch, mel_channels, time//hop_length)
        
        # Upsample mel spectrogram to match audio resolution
        mel_upsampled = F.interpolate(mel_spectrogram, size=audio.size(2), mode='nearest')
        cond = self.cond_net(mel_upsampled)
        
        x = self.input_proj(audio)
        
        skip_connections = []
        for block in self.dilated_blocks:
            x, skip = block(x + cond)
            skip_connections.append(skip)
            
        # Combine skip connections
        x = torch.stack(skip_connections).sum(dim=0)
        x = self.out_net(x)
        
        return x

In [5]:
class VoiceCloningSystem(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.speaker_encoder = SpeakerEncoder()
        self.text_encoder = TextEncoder(vocab_size)
        self.decoder = Decoder()
        self.vocoder = WaveNetVocoder()
        
    def clone_voice(self, text, reference_audio):
        # Extract speaker embedding from reference
        speaker_embedding = self.speaker_encoder(reference_audio)
        
        # Encode text
        text_encoded = self.text_encoder(text)
        
        # Generate mel spectrogram
        mel_spectrogram, _ = self.decoder(text_encoded, speaker_embedding)
        
        # Generate audio
        with torch.no_grad():
            # Initialize with noise
            audio = torch.randn(1, 1, mel_spectrogram.size(1) * 256)
            audio = self.vocoder(audio, mel_spectrogram.transpose(1, 2))
            
        return audio

In [6]:
def train_voice_cloning():
    # Three-phase training:
    
    # Phase 1: Train speaker encoder
    # Use speaker verification loss (GE2E or similar)
    
    # Phase 2: Train TTS system (fixed speaker encoder)
    # Use L1 loss for mel spectrograms
    
    # Phase 3: Train vocoder
    # Use cross-entropy for quantized audio
    
    # Phase 4: Fine-tune entire system
    pass

# Loss functions
def speaker_verification_loss(embeddings, labels):
    # Generalized end-to-end loss
    # Implement similarity-based loss that pushes same speakers together
    # and different speakers apart
    pass

def mel_loss(predicted_mel, target_mel):
    return F.l1_loss(predicted_mel, target_mel)

def vocoder_loss(predicted_audio, target_audio):
    return F.cross_entropy(predicted_audio, target_audio)